# webbot 事前学習(Google Colab)

日本語Wikipedia(hpprc/jawiki)で webbot のGPTを事前学習します。上から順に実行してください。

- メニューの「ランタイム」→「ランタイムのタイプを変更」で **T4 GPU** を選んでから始めます
- データとチェックポイントは Googleドライブの `MyDrive/webbot/` に保存します(トークン列のファイル 約1.6GB、チェックポイント 約0.5GB)
- **接続が切れたら**: もう一度上から順に実行してください。データの準備は済んでいれば飛ばし、学習はチェックポイントの続きから再開します(中断しなかった場合と同じ結果になります)

## 1. 設定

In [ ]:
SHARDS = list(range(22))      # 使う jawiki のシャード(全22、合計11.7GB)
TRAIN_TOKENS = 400_000_000    # 学習で読むトークン数(全データは約8億)
D_MODEL, N_LAYERS, N_HEADS = 512, 8, 8
SEQ_LEN, BATCH = 1024, 32     # 1ステップ = 32,768トークン
LR, WARMUP = 6e-4, 500
EVAL_INTERVAL = 500           # この間隔で検証とチェックポイント保存
COMPILE = False               # True で torch.compile(速くなる見込みだが未検証)
WANDB_PROJECT = None          # 例: "webbot"(下の W&B のセルも参照)

DRIVE = "/content/drive/MyDrive/webbot"
DATA_DRIVE = f"{DRIVE}/data"  # tokenizer.json, train.bin, val.bin
RUN_DRIVE = f"{DRIVE}/runs/base"
DATA_LOCAL = "/content/data"  # 学習中はローカルディスクのコピーを読む
STEPS = TRAIN_TOKENS // (SEQ_LEN * BATCH)
print("steps:", STEPS)


def model_args():
    return (f"--d-model {D_MODEL} --n-layers {N_LAYERS} --n-heads {N_HEADS} "
            f"--max-seq-len {SEQ_LEN} --seq-len {SEQ_LEN} --batch-size {BATCH}"
            + (" --compile" if COMPILE else ""))

## 2. GPUとドライブ

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

## 3. コードの取得とインストール

Rust版のBPE(トークナイザーの学習とエンコードが約25倍速い)もビルドします。数分かかります。

In [ ]:
import os
if os.path.exists("/content/webbot"):
    !git -C /content/webbot pull -q
else:
    !git clone -q https://github.com/KoRo2222/tiny-gpt-librarian /content/webbot
%cd /content/webbot
!pip -q install pyarrow wandb maturin
if not os.path.exists("/root/.cargo/bin/cargo"):
    !curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
!pip -q install ./rust
!python -c "import webbot_bpe; print('Rust BPE: ok')"

## 4. データの準備(初回だけ)

Hugging Face からシャードをColabにダウンロードし、トークナイザー(BPE、語彙16,000)を学習して、トークン列のファイルをドライブに書きます。全22シャードで30分前後の見込みです。

In [ ]:
from pathlib import Path
if (Path(DATA_DRIVE) / "val.bin.json").exists():
    print("prepared data found in", DATA_DRIVE)
else:
    from huggingface_hub import hf_hub_download
    raw = Path("/content/raw/jawiki")
    raw.mkdir(parents=True, exist_ok=True)
    for i in SHARDS:
        name = f"data/train-{i:05d}-of-00022.parquet"
        hf_hub_download("hpprc/jawiki", name, repo_type="dataset", local_dir="/content/raw/jawiki_hf")
        os.replace(f"/content/raw/jawiki_hf/{name}", raw / Path(name).name)
        print("downloaded", name, flush=True)
    !python scripts/prepare_data.py --raw /content/raw/jawiki --out {DATA_DRIVE} --workers 2

## 5. データをローカルディスクへコピー(接続のたびに)

In [ ]:
!mkdir -p {DATA_LOCAL} && rsync -a --info=progress2 {DATA_DRIVE}/ {DATA_LOCAL}/
!cat {DATA_LOCAL}/train.bin.json; echo; cat {DATA_LOCAL}/val.bin.json

## (任意)Weights & Biases

学習の経過をブラウザで見たいときだけ。wandb.ai のアカウント(無料)で `wandb login` し、`WANDB_MODE = "online"` にします。使わなければ、記録は `RUN_DRIVE/wandb/` にオフラインで残ります。

In [ ]:
WANDB_MODE = "offline"
# !wandb login
# WANDB_MODE = "online"

## 6. 試運転

同じ設定で50ステップだけ学習し、GPUで動くことと1ステップの時間を確かめます(本番のチェックポイントには触れません)。

In [ ]:
import re, shutil, time
shutil.rmtree("/content/smoke", ignore_errors=True)
cmd = (f"python scripts/pretrain.py --data {DATA_LOCAL} --out /content/smoke --steps 50 --warmup 10 "
       f"--eval-interval 50 --eval-batches 2 --sample-tokens 0 {model_args()}")
!{cmd}
log = open("/content/smoke/train.log", encoding="utf-8").read()
secs = float(re.findall(r"\| ([0-9.]+)s$", log, re.M)[-1])
print(f"{secs / 50:.2f} s/step -> {STEPS} steps take about {secs / 50 * STEPS / 3600:.1f} hours")

## 7. 学習

途中で接続が切れたら、1→2→3→5→7 の順に実行し直すと続きから再開します。

In [ ]:
cmd = (f"python scripts/pretrain.py --data {DATA_LOCAL} --out {RUN_DRIVE} --resume --steps {STEPS} "
       f"--warmup {WARMUP} --lr {LR} --eval-interval {EVAL_INTERVAL} {model_args()}")
if WANDB_PROJECT:
    cmd += f" --wandb-project {WANDB_PROJECT} --wandb-mode {WANDB_MODE}"
!{cmd}

## 8. 結果

`RUN_DRIVE/train.log` に検証損失の推移と生成のサンプルがあります。手元で使うときは、ドライブから次の2つをダウンロードして同じフォルダーに置きます(トークナイザーはこのノートブックで学習したものを使います)。

- `MyDrive/webbot/runs/base/checkpoint.pt`
- `MyDrive/webbot/data/tokenizer.json`

In [ ]:
!tail -n 20 {RUN_DRIVE}/train.log